In [6]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score, confusion_matrix

df = pd.read_csv("./human_oracle_cross_validation.csv")

# -----------------------------
# 1. Copy dataframe
# -----------------------------
df_eval = df.copy()

# -----------------------------
# 2. Clean + convert labels safely
# -----------------------------
df_eval["source_query_project"] = pd.to_numeric(
    df_eval["source_query_project"],
    errors="coerce"
)

df_eval["cross_val_category"] = pd.to_numeric(
    df_eval["cross_val_category"],
    errors="coerce"
)

# -----------------------------
# 3. Keep only valid classes
# -----------------------------
valid_labels = {1, 2, 3, 4}

df_eval = df_eval[
    df_eval["source_query_project"].isin(valid_labels) &
    df_eval["cross_val_category"].isin(valid_labels)
].copy()

# -----------------------------
# 4. Sanity check (IMPORTANT)
# -----------------------------
print(f"Rows used for evaluation: {len(df_eval)}")

if len(df_eval) == 0:
    raise ValueError(
        "No valid rows left after filtering. "
        "Check label formats in your dataframe."
    )

# -----------------------------
# 5. Map to binary classes
# -----------------------------
# 1,2 → Positive (1)
# 3,4 → Negative (0)

label_map = {
    1: 1,
    2: 1,
    3: 0,
    4: 0
}

y_true = df_eval["source_query_project"].map(label_map)
y_pred = df_eval["cross_val_category"].map(label_map)

# Remove any remaining NaNs just in case
mask = y_true.notna() & y_pred.notna()
y_true = y_true[mask].astype(int)
y_pred = y_pred[mask].astype(int)

# -----------------------------
# 6. Metrics
# -----------------------------
kappa = cohen_kappa_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

# -----------------------------
# 7. Output results
# -----------------------------
print("\n=== Evaluation Results ===")
print(f"Cohen's Kappa: {kappa:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")

print("\nConfusion Matrix:")
print(cm)

Rows used for evaluation: 1130

=== Evaluation Results ===
Cohen's Kappa: 0.8926
Precision:     0.9844
Recall:        0.9392

Confusion Matrix:
[[379  11]
 [ 45 695]]


In [5]:
from sklearn.metrics import cohen_kappa_score
import pandas as pd
df = pd.read_csv("./human_oracle_cross_validation.csv")

# ---- clean labels ----
y1 = df["human_agreement"].astype(str).str.strip()
y2 = df["agreement_flag"].astype(str).str.strip()

# ---- compute Cohen's Kappa ----
kappa = cohen_kappa_score(y1, y2)

print("Cohen's Kappa:", kappa)

Cohen's Kappa: 0.8162861384039019


In [11]:
#Load the production dataset:

import pandas as pd

df = pd.read_csv("downsized_compliance_empirical_dataset.csv")
print(df.shape)
df.head()

(197557, 10)


,organization_name,hash,source_method_name,sink_method_name,source_repository_url,sink_repository_url,source_file_location,sink_file_location,source_query_project,violation
0,Accenture,064a8ca19bb9f59ebb28c8f28a847538,byte,byte,https://github.com/isislovecruft/scripts/blob/...,https://github.com/Accenture/mf_inspector/blob...,include/dstevens/pdfid.py:101,./modules/pdfid/pdfid.py:127,5,Undetermined: LGPL-3.0-only with the Unknown (...
1,Accenture,06f999eb12cc905a8d7b3e53643c7fa2,ioctl_GWINSZ,ioctl_GWINSZ,https://github.com/ivolo/animals/blob/0bcaa54f...,https://github.com/Accenture/pandas/blob/maste...,examples/loading.py:15,./pandas/util/terminal.py:92,4,High Risk - Unknown/Unlabeled source entering ...
2,Accenture,085b83d9a7c9b07dc55154fdf8743167,minor_xs,minor_xs,https://github.com/pandas-dev/pandas/blob/fde2...,https://github.com/Accenture/pandas/blob/maste...,pandas/core/sparse.py:1681,./pandas/sparse/panel.py:476,3,Custom incompatible with BSD-3-Clause
3,Accenture,08a1142684490c7a61a34594fbf06708,adjust_transform_for_image,adjust_transform_for_image,https://github.com/fizyr/keras-retinanet/blob/...,https://github.com/Accenture/AIR/blob/master/k...,keras_retinanet/utils/image.py:64,./keras_retinanet/keras_retinanet/utils/image....,1,Sink is following same license Apache-2.0
4,Accenture,0a00bf92690aa143538be9a7c24d79cd,chop_data,chop_data,https://github.com/pydata/pandas-datareader/bl...,https://github.com/Accenture/pandas/blob/maste...,pandas_datareader/data.py:927,./pandas/io/data.py:945,5,Undetermined: Custom with the Custom (Provenan...


In [13]:
#Load the human validation dataset:

validation = pd.read_csv("human_oracle_cross_validation.csv")
print(validation.shape)
validation.head()

(1130, 22)


,organization_name,project_type,hash,source_method_name,sink_method_name,source_repository_url,sink_repository_url,source_file_location,sink_file_location,source_query_project,...,source_license,sink_license,source_repo_level_license,sink_repo_level_license,cross_val_category,cross_val_message,agreement_flag,human_annotator_1,human_annotator_2,human_agreement
0,Google,1,5cfbd210d0c6557278295eef43b81e54,ReadFromSourceProperties,ReadFromSourceProperties,https://github.com/google/play-instant-unity-p...,https://github.com/google/play-unity-plugins/b...,GooglePlayInstant/Editor/GooglePlayServices/An...,./GooglePlayPlugins/com.google.android.appbund...,3,...,Apache-2.0,Custom,Apache-2.0,NOASSERTION,3,Custom incompatible with Apache-2.0,Agreed,3,3,Agreed
1,Microsoft,1,8f60c82eafadd32e3c8b96a2eddd5566,softfloat_approxRecip32_1,softfloat_approxRecip32_1,https://github.com/urbit/urbit/blob/377f7c3328...,https://github.com/microsoft/optee_os/blob/mas...,outside/softfloat-3/source/s_approxRecip32_1.c:39,./lib/libutils/isoc/arch/arm/softfloat/source/...,3,...,MIT,Custom,MIT,NOASSERTION,3,Custom incompatible with MIT,Agreed,3,3,Agreed
2,Nvidia,1,36b1bc91aec958dcec2efe6f9db1c301,subdeviceCtrlCmdNvdGetDump_IMPL,subdeviceCtrlCmdNvdGetDump_IMPL,https://github.com/microsoft/vattention/blob/e...,https://github.com/NVIDIA/open-gpu-kernel-modu...,./nvidia-vattn-uvm-driver/src/nvidia/kernel/nv...,./src/nvidia/src/kernel/gpu/subdevice/subdevic...,3,...,MIT,Custom,MIT,NOASSERTION,3,Custom incompatible with MIT,Agreed,3,3,Agreed
3,Microsoft,1,5f4303c729b34d9d2f0bec0a5e8978a0,job,job,https://github.com/malllabiisc/EmbedKGQA/blob/...,https://github.com/microsoft/HittER/blob/main/...,./kge/model/kge_model.py:578,./kge/model/kge_model.py:450,2,...,Apache-2.0,MIT,Apache-2.0,MIT,2,MIT compatible with Apache-2.0,Agreed,2,2,Agreed
4,Microsoft,1,0cea33e695ff78e2fe8780ae13e97eeb,BuildMaterials,BuildMaterials,https://github.com/aminuldidar/DirectX-Graphic...,https://github.com/microsoft/DirectX-Graphics-...,./MiniEngine/Model/ModelConvert.cpp:253,./MiniEngine/Model/ModelConvert.cpp:253,3,...,GPL-3.0-only,MIT,No License File Found,MIT,3,MIT incompatible with Custom,Agreed,3,3,Agreed
